# 06 – Ensemble: LightGBM + LSTM

Kombinujemo predikcije dva modela:
- **LightGBM + Prophet** (val RMSLE = 0.3695)
- **LSTM** (val RMSLE = 0.4140)

Strategije:
1. Jednostavan prosjek (50/50)
2. Težinski prosjek – grid search za optimalan α
3. Ensemble u log-prostoru
4. Poređenje svih pristupa

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import pickle
import warnings
warnings.filterwarnings('ignore')

PROCESSED = '../data/processed/'
MODELS    = '../models/'

def rmsle(actual, pred):
    pred = np.clip(pred, 0, None)
    return np.sqrt(np.mean((np.log1p(pred) - np.log1p(actual))**2))

print('Spreman.')

## 1. Učitaj LightGBM predikcije

In [ ]:
# Učitaj model
with open(MODELS + 'lgbm_prophet.pkl', 'rb') as f:
    lgbm_data = pickle.load(f)

lgbm_model    = lgbm_data['model']
lgbm_features = lgbm_data['features']
print(f'LightGBM features: {len(lgbm_features)}')
print(f'Best iteration:    {lgbm_model.best_iteration}')

# Val set
val_df = pd.read_parquet(PROCESSED + 'val_features.parquet')
print(f'Val shape: {val_df.shape}')

X_val = val_df[lgbm_features]
y_val = np.expm1(val_df['sales_log'])

lgbm_pred_log   = lgbm_model.predict(X_val, num_iteration=lgbm_model.best_iteration)
lgbm_pred_sales = np.expm1(lgbm_pred_log).clip(min=0)

lgbm_rmsle = rmsle(y_val, lgbm_pred_sales)
print(f'\nLightGBM RMSLE: {lgbm_rmsle:.4f}')

## 2. Učitaj LSTM predikcije

In [ ]:
lstm_val = pd.read_parquet(MODELS + 'lstm_val_preds.parquet')
print(f'LSTM val shape: {lstm_val.shape}')
print(lstm_val.head(3).to_string())

lstm_rmsle = rmsle(lstm_val['actual_sales'], lstm_val['lstm_pred'])
print(f'\nLSTM RMSLE: {lstm_rmsle:.4f}')

## 3. Spoji predikcije

In [ ]:
# Dodaj LightGBM predikcije u val_df
val_df = val_df.copy()
val_df['lgbm_pred']     = lgbm_pred_sales
val_df['lgbm_pred_log'] = lgbm_pred_log

# Merge sa LSTM predikcijama
merged = val_df[['date', 'store_nbr', 'family', 'sales', 'lgbm_pred', 'lgbm_pred_log']].merge(
    lstm_val[['date', 'store_nbr', 'family', 'lstm_pred', 'lstm_pred_log']],
    on=['date', 'store_nbr', 'family'],
    how='inner'
)

print(f'Merged shape: {merged.shape}')
print(f'Datumi: {merged["date"].min().date()} → {merged["date"].max().date()}')
print(merged.head(3)[['date','store_nbr','family','sales','lgbm_pred','lstm_pred']].to_string())

## 4. Jednostavan prosjek (50/50)

In [ ]:
actual = merged['sales'].values

# Prosjek u sales prostoru
avg_pred = (merged['lgbm_pred'].values + merged['lstm_pred'].values) / 2
avg_rmsle = rmsle(actual, avg_pred)

# Prosjek u log prostoru (geometrijska sredina u originalnom prostoru)
avg_log_pred = np.clip(
    np.expm1((merged['lgbm_pred_log'].values + merged['lstm_pred_log'].values) / 2), 0, None
)
avg_log_rmsle = rmsle(actual, avg_log_pred)

print('Jednostavan prosjek (50/50):')
print(f'  Sales prostor:  RMSLE = {avg_rmsle:.4f}')
print(f'  Log prostor:    RMSLE = {avg_log_rmsle:.4f}')
print()
print('Referentne vrijednosti:')
print(f'  LightGBM:       RMSLE = {lgbm_rmsle:.4f}')
print(f'  LSTM:           RMSLE = {lstm_rmsle:.4f}')

## 5. Grid search – optimalan težinski prosjek

In [ ]:
alphas = np.linspace(0, 1, 101)  # α = udio LightGBM

rmsle_sales = []
rmsle_log   = []

lgbm_p = merged['lgbm_pred'].values
lstm_p = merged['lstm_pred'].values
lgbm_l = merged['lgbm_pred_log'].values
lstm_l = merged['lstm_pred_log'].values

for a in alphas:
    # Sales prostor
    pred_s = a * lgbm_p + (1 - a) * lstm_p
    rmsle_sales.append(rmsle(actual, pred_s))
    # Log prostor
    pred_l = np.expm1(a * lgbm_l + (1 - a) * lstm_l).clip(min=0)
    rmsle_log.append(rmsle(actual, pred_l))

best_alpha_s = alphas[np.argmin(rmsle_sales)]
best_rmsle_s = min(rmsle_sales)
best_alpha_l = alphas[np.argmin(rmsle_log)]
best_rmsle_l = min(rmsle_log)

print(f'Optimalan α (sales prostor): {best_alpha_s:.2f}  →  RMSLE = {best_rmsle_s:.4f}')
print(f'Optimalan α (log prostor):   {best_alpha_l:.2f}  →  RMSLE = {best_rmsle_l:.4f}')
print()
print('Napomena: α je udio LightGBM (α=1 → samo LightGBM, α=0 → samo LSTM)')

In [ ]:
fig, ax = plt.subplots(figsize=(10, 4))

ax.plot(alphas, rmsle_sales, label='Ensemble (sales prostor)', color='steelblue')
ax.plot(alphas, rmsle_log,   label='Ensemble (log prostor)',   color='darkorange')
ax.axhline(lgbm_rmsle, color='green',  linestyle='--', alpha=0.7, label=f'LightGBM ({lgbm_rmsle:.4f})')
ax.axhline(lstm_rmsle, color='red',    linestyle='--', alpha=0.7, label=f'LSTM ({lstm_rmsle:.4f})')
ax.axvline(best_alpha_s, color='steelblue', linestyle=':', alpha=0.8)
ax.axvline(best_alpha_l, color='darkorange', linestyle=':', alpha=0.8)

ax.set_xlabel('α (udio LightGBM)')
ax.set_ylabel('RMSLE')
ax.set_title('Grid search – optimalni težinski prosjek')
ax.legend()
plt.tight_layout()
plt.show()

## 6. Poređenje modela

In [ ]:
# Finalne predikcije najboljeg ensembla
best_ensemble_pred = (
    best_alpha_l * merged['lgbm_pred_log'].values + (1 - best_alpha_l) * merged['lstm_pred_log'].values
)
best_ensemble_sales = np.clip(np.expm1(best_ensemble_pred), 0, None)

results = {
    'Naive baseline (lag_7)':       0.5690,
    'LSTM':                         lstm_rmsle,
    'Ensemble 50/50 (log)':         avg_log_rmsle,
    f'Ensemble α={best_alpha_l:.2f} (log)': best_rmsle_l,
    'LightGBM + Prophet':           lgbm_rmsle,
}

print('=' * 45)
print(f'{"Model":<35} {"RMSLE":>8}')
print('=' * 45)
for name, score in sorted(results.items(), key=lambda x: x[1], reverse=True):
    marker = ' ◄ BEST' if score == min(results.values()) else ''
    print(f'{name:<35} {score:>8.4f}{marker}')
print('=' * 45)

improvement = (lgbm_rmsle - best_rmsle_l) / lgbm_rmsle * 100
print(f'\nEnsemble poboljšanje vs LightGBM: {improvement:+.2f}%')

## 7. Vizualizacija – dnevne predikcije

In [ ]:
merged['ensemble_pred'] = best_ensemble_sales

daily = merged.groupby('date').agg(
    actual    = ('sales',         'sum'),
    lgbm      = ('lgbm_pred',     'sum'),
    lstm      = ('lstm_pred',     'sum'),
    ensemble  = ('ensemble_pred', 'sum'),
).reset_index()

fig, axes = plt.subplots(2, 1, figsize=(12, 8))

# Gornji panel: dnevne predikcije
ax = axes[0]
ax.plot(daily['date'], daily['actual'],   label='Stvarna prodaja', color='black',      linewidth=2)
ax.plot(daily['date'], daily['lgbm'],     label='LightGBM',        color='green',      linewidth=1.5, linestyle='--')
ax.plot(daily['date'], daily['lstm'],     label='LSTM',             color='red',        linewidth=1.5, linestyle='--')
ax.plot(daily['date'], daily['ensemble'], label='Ensemble',         color='darkorange', linewidth=2,   linestyle='-.')
ax.set_title('Ukupna dnevna prodaja – val set')
ax.set_ylabel('Sales')
ax.legend()

# Donji panel: relativna greška
ax2 = axes[1]
ax2.plot(daily['date'], (daily['lgbm']     - daily['actual']) / daily['actual'] * 100, label='LightGBM',  color='green',      linewidth=1.5)
ax2.plot(daily['date'], (daily['lstm']     - daily['actual']) / daily['actual'] * 100, label='LSTM',      color='red',        linewidth=1.5)
ax2.plot(daily['date'], (daily['ensemble'] - daily['actual']) / daily['actual'] * 100, label='Ensemble',  color='darkorange', linewidth=2)
ax2.axhline(0, color='black', linewidth=0.8)
ax2.set_title('Relativna greška (%)')
ax2.set_ylabel('Greška (%)')
ax2.legend()

plt.tight_layout()
plt.show()

## 8. RMSLE po kategoriji – ensemble vs LightGBM

In [ ]:
family_results = merged.groupby('family').apply(
    lambda g: pd.Series({
        'lgbm_rmsle':     rmsle(g['sales'], g['lgbm_pred']),
        'lstm_rmsle':     rmsle(g['sales'], g['lstm_pred']),
        'ensemble_rmsle': rmsle(g['sales'], g['ensemble_pred']),
    })
).reset_index()

family_results['ensemble_vs_lgbm'] = family_results['ensemble_rmsle'] - family_results['lgbm_rmsle']
family_results = family_results.sort_values('ensemble_vs_lgbm')

# Kategorije gdje ensemble pomaže najviše
print('Top 10 kategorija gdje ensemble poboljšava LightGBM:')
print(family_results.head(10)[['family', 'lgbm_rmsle', 'lstm_rmsle', 'ensemble_rmsle', 'ensemble_vs_lgbm']].to_string(index=False))

print('\nKategorije gdje ensemble šteti:')
worse = family_results[family_results['ensemble_vs_lgbm'] > 0]
print(f'  {len(worse)} od {len(family_results)} kategorija')
if len(worse) > 0:
    print(worse.tail(5)[['family', 'lgbm_rmsle', 'lstm_rmsle', 'ensemble_rmsle', 'ensemble_vs_lgbm']].to_string(index=False))

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))

x = np.arange(len(family_results))
w = 0.35
ax.bar(x - w/2, family_results['lgbm_rmsle'],     width=w, label='LightGBM', color='steelblue', alpha=0.8)
ax.bar(x + w/2, family_results['ensemble_rmsle'], width=w, label='Ensemble',  color='darkorange', alpha=0.8)

ax.set_xticks(x)
ax.set_xticklabels(family_results['family'], rotation=45, ha='right', fontsize=8)
ax.set_ylabel('RMSLE')
ax.set_title('RMSLE po kategoriji: LightGBM vs Ensemble')
ax.legend()
plt.tight_layout()
plt.show()

## 9. Sačuvaj ensemble predikcije

In [ ]:
ensemble_config = {
    'alpha_lgbm':   best_alpha_l,
    'alpha_lstm':   1 - best_alpha_l,
    'blend_space':  'log',
    'val_rmsle':    best_rmsle_l,
    'lgbm_rmsle':   lgbm_rmsle,
    'lstm_rmsle':   lstm_rmsle,
}

with open(MODELS + 'ensemble_config.pkl', 'wb') as f:
    pickle.dump(ensemble_config, f)

merged[['date', 'store_nbr', 'family', 'sales', 'lgbm_pred', 'lstm_pred', 'ensemble_pred']].to_parquet(
    MODELS + 'ensemble_val_preds.parquet', index=False
)

print('Sačuvano:')
print(f'  ensemble_config.pkl     → α_lgbm={best_alpha_l:.2f}, α_lstm={1-best_alpha_l:.2f}')
print(f'  ensemble_val_preds.parquet')
print()
print('Finalni rezultati:')
print(f'  Naive baseline: RMSLE = 0.5690')
print(f'  LSTM:           RMSLE = {lstm_rmsle:.4f}')
print(f'  LightGBM:       RMSLE = {lgbm_rmsle:.4f}')
print(f'  Ensemble:       RMSLE = {best_rmsle_l:.4f}')